In [1]:
import polars as pl
from openhexa.toolbox.dhis2 import DHIS2
from openhexa.sdk import DHIS2Connection, current_run, parameter, pipeline, workspace
import regex as re

In [2]:
con = DHIS2Connection(
    url="https://dhis2.minisante.gov.bi", username="admin", password="zu]T2f9&RPJAkn>CRLNu:@"
)
dhis2 = DHIS2(connection=con)

In [3]:
def get_data_elements(dhis2: DHIS2, filters: list[str] | None = None) -> pl.DataFrame:
    """Extract data elements metadata.

    Parameters
    ----------
    dhis2 : DHIS2
        DHIS2 instance.
    filters : list[str], optional
        DHIS2 query filter expressions.

    Returns
    -------
    pl.DataFrame
        Dataframe containing data elements metadata with the following columns: id, name, value_type.
    """
    meta = dhis2.meta.data_elements(fields="id,name,valueType", filters=filters, pageSize=100)
    schema = {"id": str, "name": str, "valueType": str}
    df = pl.DataFrame(meta, schema=schema)
    return df.select("id", "name", pl.col("valueType").alias("value_type"))

In [4]:
des_all = get_data_elements(dhis2)

In [5]:
des = des_all.filter(
    (pl.col("name").str.contains("FBP"))
    & ~(
        pl.col("name").str.contains(
            "Couverture|Ecart|montant|Montant|Population|Validee| Val |Verif Indiv|Verif|Tarif|Cible|Pts Obtenus|Pts Possible|Pourcentage écart"
        )
    )
)

In [6]:
des_unmapped = des.filter(
    ~(
        pl.col("name").str.contains(
            "QUANT17 |QUANT30 |QUANT21 |QUANT02 |QUANT01 |QUANT05 |QUANT09 |QUANT56 |QUANT55 |QUANT51 |QUANT52 |QUANT23 |QUANT17_1 |QUANT17_HC |QUANT08_HC "
        )
    )
    & ~(
        pl.col("name").str.contains(
            "QUANT13_HC |QUANT12_HC |QUANT15_HC |QUANT09_HC |QUANT04_HC |QUANT01_HC |QUANT02_HC |QUANT03_HC |QUANT10_HC |QUANT06_HC |QUANT11_HC "
        )
    )
    & ~(
        pl.col("name").str.contains(
            "QUANT14_HC |QUANT7_CNPK |QUANT14_CNPK |QUANT11_CNPK |QUANT4_CNPK |QUANT3_CNPK |QUANT13_CNPK |QUANT12_CNPK |QUANT2_CNPK |QUANT1_CNPK "
        )
    )
    & ~(
        pl.col("name").str.contains(
            "QUANT07_01_HC |QUANT28 |QUANT53 |QUANT25 |QUANT24 |QUANT53 |QUANT29 |QUANT27 |QUANT51_1 |QUANT16 |QUANT50_1 "
        )
    )
)

In [7]:
now = des.filter(
    (pl.col("name").str.contains("QUANT50_1")) & ~(pl.col("name").str.contains("QUANT16_HC"))
)
with pl.Config(fmt_str_lengths=200):
    display(now)

id,name,value_type
str,str,str
"""EpT3PTOrHLX""","""FBP - Declaree Indiv - QUANT50_1 Nuit hospitalisation""","""NUMBER"""
"""vFSEGOPawcE""","""FBP - Decl - QUANT50_1 Nuit hospitalisation""","""NUMBER"""
"""G8g9ZAL3YRA""","""FBP - Nb cas remboursement complets Indiv - QUANT50_1 Nuit hospitalisation""","""NUMBER"""
"""QJpxZfpyziK""","""FBP - Nb cas remboursement complets - QUANT50_1 Nuit hospitalisation""","""NUMBER"""
"""hk4SPJzcZDM""","""FBP - Nb cas remboursement MFP(80%) Indiv - QUANT50_1 Nuit hospitalisation""","""NUMBER"""
"""OzQ3EN6o16r""","""FBP - Nb cas remboursement MFP(80%) - QUANT50_1 Nuit hospitalisation""","""NUMBER"""
"""ZlRzMPsXMD2""","""FBP - Tot - QUANT50_1 Nuit hospitalisation""","""NUMBER"""


In [8]:
des_unmapped_decl_indiv = (
    des_unmapped.filter(pl.col("name").str.contains("Declaree"))
    .select(pl.col("name"))
    .to_series()
    .to_list()
)
des_unmapped_decl = (
    des_unmapped.filter(pl.col("name").str.contains("Decl "))
    .select(pl.col("name"))
    .to_series()
    .to_list()
)

In [9]:
des_unmapped_decl_indiv

["FBP - Declaree Indiv - QUANT03 Journée d'hospitalisation >= 5 ans",
 "FBP - Declaree Indiv - QUANT04 Journée d'hospitalisation < 5 ans",
 "FBP - Declaree Indiv - QUANT06 Référence et patient arrivé à l'hopital",
 'FBP - Declaree Indiv - QUANT07 Enfants complétement vaccinés (VAR2)',
 'FBP - Declaree Indiv - QUANT08 Femmes enceintes VAT completement vacciné',
 "FBP - Declaree Indiv - QUANT10 Prise en charge du nouveau né d'une femme VIH+",
 'FBP - Declaree Indiv - QUANT11 Nombre de nouveaux cas sous ARV',
 'FBP - Declaree Indiv - QUANT12 Nombre de clients ARV suivi semestriellement',
 'FBP - Declaree Indiv - QUANT13 Cas des IST traitées',
 'FBP - Declaree Indiv - QUANT14 Dépistage des cas TBC positifs par mois',
 'FBP - Declaree Indiv - QUANT15 Nombre de cas TBC traités pendant un semestre et guéris',
 'FBP - Declaree Indiv - QUANT18 FP: Tot. Nouveaux + Anciennes Acceptantes',
 'FBP - Declaree Indiv - QUANT19 FP: Implants et DIU',
 'FBP - Declaree Indiv - QUANT20 Consultation prénatal

In [10]:
des_unmapped_decl

['FBP - Decl - GASC_QUANT_01 Cas de malnutrition aigue severe(MAS), depistés, referés et confirmé par le CDS',
 'FBP - Decl - GASC_QUANT_02 Cas de tuberculose suspectés par les Asc,référés et confirmés par le CDS',
 'FBP - Decl - GASC_QUANT_03 Femme enceinte référée pour accouchement et arrivée au CDS',
 'FBP - Decl - GASC_QUANT_04 Femme enceinte,avec risque, accompagnée et arrivée au CDS pour accouchement',
 'FBP - Decl - GASC_QUANT_05 Femme avec fistule obstétricale suspectée,référée et arrivé au CDS',
 "FBP - Decl - GASC_QUANT_06 Nouvelle acceptante d'une méthode de planification familiale référée et arrivé au CDS",
 'FBP - Decl - GASC_QUANT_07 Femme enceinte référée pour une première consultation prénatale précoce (CPN1) au premier trimestre de la grossesse',
 "FBP - Decl - GASC_QUANT_08 Mère référée pour consultation post natale et arrivée au CDS dans les premiers jours suivant l'accouchement",
 'FBP - Decl - GASC_QUANT_09 Femme enceinte référée pour dépistage du VIH/PTME',
 'FBP 

In [11]:
codes_decl_indiv_unmaped = [
    m.group(1) for item in des_unmapped_decl_indiv if (m := re.search(r"Indiv - (\w+)", item))
]

codes_decl_unmaped = [
    m.group(1) for item in des_unmapped_decl if (m := re.search(r"Decl - (\w+)", item))
]

In [14]:
codes_decl_indiv_unmaped

['QUANT03',
 'QUANT04',
 'QUANT06',
 'QUANT07',
 'QUANT08',
 'QUANT10',
 'QUANT11',
 'QUANT12',
 'QUANT13',
 'QUANT14',
 'QUANT15',
 'QUANT18',
 'QUANT19',
 'QUANT20',
 'QUANT22',
 'QUANT23_1',
 'QUANT26',
 'QUANT31',
 'QUANT32',
 'QUANT34',
 'QUANT35',
 'QUANT36',
 'QUANT37',
 'QUANT38',
 'QUANT50',
 'QUANT57']